A class is like a blueprint. it defines what things will look like and what they can do meanwhile an oject is an instance of a class. eg if we have class car then mercedes is an object

In [1]:
class Product:

    def __init__(self, name, category, unit_price, quantity_in_stock):
        self.name = name
        self.category = category
        self.unit_price = unit_price              
        self.quantity_in_stock = quantity_in_stock

    def stock_value(self):
        return self.unit_price * self.quantity_in_stock
        

In [2]:
rice = Product("Rice 25kg", "Grains", 19000, 40)
palm_oil = Product("Palm Oil 5L", "Oils", 6500, 25)
bread = Product("Bread (loaf)", "Bakery", 600, 60)

for item in (rice, palm_oil, bread):
    print(item.name, "->", f"{item.stock_value():,}", "FCFA")

Rice 25kg -> 760,000 FCFA
Palm Oil 5L -> 162,500 FCFA
Bread (loaf) -> 36,000 FCFA


In [3]:
# changing ONE object and checking that the others don't care
print("before:", rice.unit_price, palm_oil.unit_price, bread.unit_price)

rice.unit_price = 20000

print("after: ", rice.unit_price, palm_oil.unit_price, bread.unit_price)

before: 19000 6500 600
after:  20000 6500 600


2.Adding Behaviour

A second method, `sell()`, that changes an attribute instead of just reading it.
(Re-running the class cell replaces the old `Product`, so the objects get created again below.)

In [4]:
class Product:
    # same class as before, but now with a sell() method added

    def __init__(self, name, category, unit_price, quantity_in_stock):
        self.name = name
        self.category = category
        self.unit_price = unit_price
        self.quantity_in_stock = quantity_in_stock

    def stock_value(self):
        return self.unit_price * self.quantity_in_stock

    def sell(self, units):
        if units > self.quantity_in_stock:
            print(f"Can't sell {units} of {self.name}, only {self.quantity_in_stock} left")
            return
        self.quantity_in_stock -= units

In [ ]:
rice = Product("Rice 25kg", "Grains", 19000, 40)
bread = Product("Bread (loaf)", "Bakery", 600, 60)

print("rice stock before:", rice.quantity_in_stock)
rice.sell(12)
print("rice stock after: ", rice.quantity_in_stock)

# bread shouldn't have moved
print("bread stock still:", bread.quantity_in_stock)

# what happens if I try to sell more than we have
rice.sell(500)

rice stock before: 40
rice stock after:  28
bread stock still: 60
Can't sell 500 of Rice 25kg, only 28 left


Sell needs self because When I write rice.sell(12), Python quietly passes rice in as self. That's how the method knows whose quantity_in_stock to change: self.quantity_in_stock means rice's stock, not breads stock

3.Inheritance

Some products spoil, so they need one extra thing `Product` doesn't have: a shelf life.

In [8]:
class PerishableProduct(Product):
    # a product that goes bad, so it also needs a shelf life

    def __init__(self, name, category, unit_price, quantity_in_stock, shelf_life_days):
        super().__init__(name, category, unit_price, quantity_in_stock)
        self.shelf_life_days = shelf_life_days

In [9]:
fresh_bread = PerishableProduct("Bread (loaf)", "Bakery", 600, 60, 3)

# 1) methods it got from Product without me rewriting them
print(fresh_bread.name, "stock value:", f"{fresh_bread.stock_value():,}", "FCFA")
fresh_bread.sell(10)
print("stock after selling 10:", fresh_bread.quantity_in_stock)

# 2) the extra attribute that only the child has
print("shelf life:", fresh_bread.shelf_life_days, "days")

# and a plain Product doesn't have it
print("does rice have a shelf life?", hasattr(rice, "shelf_life_days"))

Bread (loaf) stock value: 36,000 FCFA
stock after selling 10: 50
shelf life: 3 days
does rice have a shelf life? False


4 Polymorphism

Both classes get a describe() method with the same name, but the perishable version adds its own detail.

In [11]:
class Product:
    def __init__(self, name, category, unit_price, quantity_in_stock):
        self.name = name
        self.category = category
        self.unit_price = unit_price
        self.quantity_in_stock = quantity_in_stock

    def stock_value(self):
        return self.unit_price * self.quantity_in_stock

    def sell(self, units):
        if units > self.quantity_in_stock:
            print(f"Can't sell {units} of {self.name}, only {self.quantity_in_stock} left")
            return
        self.quantity_in_stock -= units

    def describe(self):
        return f"{self.name} ({self.category}): {self.quantity_in_stock} in stock at {self.unit_price:,.0f} FCFA each"


class PerishableProduct(Product):
    def __init__(self, name, category, unit_price, quantity_in_stock, shelf_life_days):
        super().__init__(name, category, unit_price, quantity_in_stock)
        self.shelf_life_days = shelf_life_days

    def describe(self):
        base_text = super().describe()
        return f"{base_text}. Best sold within {self.shelf_life_days} days"

In [12]:
rice = Product("Rice 25kg", "Grains", 19000, 40)
fresh_bread = PerishableProduct("Bread (loaf)", "Bakery", 600, 60, 3)

products = [rice, fresh_bread]

# same loop, same method name, but different output for each object
for p in products:
    print(p.describe())

Rice 25kg (Grains): 40 in stock at 19,000 FCFA each
Bread (loaf) (Bakery): 60 in stock at 600 FCFA each. Best sold within 3 days


Polymorphism note: the loop is identical for every item, but each object runs *its own* version of `describe()`, so the same line of code gives different output depending on which kind of object it's touching.

5. Abstraction

An Inventory class that holds all the products and hides the looping behind one method.

In [14]:
class Inventory:
    def __init__(self):
        self.products = []          

    def add_product(self, product):
        self.products.append(product)

    def total_stock_value(self):
        # the looping is hidden in here
        total = 0
        for product in self.products:
            total += product.stock_value()
        return total


shop = Inventory()
shop.add_product(Product("Rice 25kg", "Grains", 19000, 40))
shop.add_product(Product("Palm Oil 5L", "Oils", 6500, 25))
shop.add_product(PerishableProduct("Bread (loaf)", "Bakery", 600, 60, 3))

# one simple call, no loop in sight
print("total stock value:", f"{shop.total_stock_value():,}", "FCFA")

total stock value: 958,500 FCFA


6. Connecting it to real libraries

A Pandas DataFrame is an object, just like my rice product. When I write df.head() or df.describe(), I'm calling methods on that DataFrame, the same way I call rice.sell(12) on rice. The library is built from classes and objects, so what I practiced here is how Pandas works underneath.